#### 本教程演示了如何使用Adversarial Debiasing算法来学习公平的分类器。

对抗性去偏见，Adversarial debiasing[1]是一种处理中技术，此方法通过学习分类器来最大限度地提高预测准确性，同时降低对手从预测中确定受保护属性的能力。这种方法可以得到一个公平的分类器，因为预测不可能携带任何可被对手利用的群体歧视信息。在这个教程中，你将了解如何使用这种算法来学习有公平性约束和无公平性约束的模型，并将它们应用于 Adult 数据集。

In [1]:
%matplotlib inline

# 加载需要的库
import sys
sys.path.append("../")
from aif360.datasets import BinaryLabelDataset
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.datasets import DuolingoDataset

from aif360.datasets import StuperDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.metrics.utils import compute_boolean_conditioning_vector

from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult, load_preproc_data_compas, load_preproc_data_german
# from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import  load_preproc_data_stuper
from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_duolingo

from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_eager_execution()

# 用于创建专家特征偏好的可视化
import pandas as pd

pip install 'aif360[LawSchoolGPA]'
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted 

#### 读取数据集和设置选项

In [2]:
# 获取数据集，进行训练集和测试集的划分
dataset_orig = load_preproc_data_duolingo()

privileged_groups = [{'ui_binary': 1}]
unprivileged_groups = [{'ui_binary': 0}]

dataset_orig_train, dataset_orig_test = dataset_orig.split([0.7], shuffle=True)

C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\aif360\algorithms\preprocessing\optim_preproc_helpers\data_preproc_functions.py:352: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ui_binary'] = df['ui_binary'].replace({'eng': 0, 'noneng': 1})
C:\Users\Gao\.conda\envs\AIF360v2\lib\site-packages\aif360\algorithms\preprocessing\optim_preproc_helpers\data_preproc_functions.py:353: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['learning_binary'] = df['learning_binary'].replace({'eng': 0, 'noneng': 1})
C:\Users\Gao\.conda\envs

In [3]:
# 打印出数据集的一些特征
display(Markdown("#### Training Dataset shape"))
print(dataset_orig_train.features.shape)
display(Markdown("#### Favorable and unfavorable labels"))
print(dataset_orig_train.favorable_label, dataset_orig_train.unfavorable_label)
display(Markdown("#### Protected attribute names"))
print(dataset_orig_train.protected_attribute_names)
display(Markdown("#### Privileged and unprivileged protected attribute values"))
print(dataset_orig_train.privileged_protected_attributes, 
      dataset_orig_train.unprivileged_protected_attributes)
display(Markdown("#### Dataset feature names"))
print(dataset_orig_train.feature_names)

#### Training Dataset shape

(734002, 7)


#### Favorable and unfavorable labels

1.0 0.0


#### Protected attribute names

['ui_binary', 'learning_binary']


#### Privileged and unprivileged protected attribute values

[array([1.]), array([1.])] [array([0.]), array([0.])]


#### Dataset feature names

['ui_binary', 'learning_binary', 'delta', 'history_seen', 'history_correct', 'session_seen', 'is_workday=1']


#### 原始训练数据的指标

In [4]:
# 原始数据集的指标
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_orig_test.mean_difference())

#### Original training dataset

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.002111
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.000662


In [5]:
min_max_scaler = MaxAbsScaler()
dataset_orig_train.features = min_max_scaler.fit_transform(dataset_orig_train.features)
dataset_orig_test.features = min_max_scaler.transform(dataset_orig_test.features)
metric_scaled_train = BinaryLabelDatasetMetric(dataset_orig_train, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
# 缩放数据集 - 验证缩放是否不会影响组标签统计数据
display(Markdown("#### Scaled dataset - Verify that the scaling does not affect the group label statistics"))
print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_train.mean_difference())
metric_scaled_test = BinaryLabelDatasetMetric(dataset_orig_test, 
                             unprivileged_groups=unprivileged_groups,
                             privileged_groups=privileged_groups)
print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_scaled_test.mean_difference())


#### Scaled dataset - Verify that the scaling does not affect the group label statistics

Train set: Difference in mean outcomes between unprivileged and privileged groups = -0.002111
Test set: Difference in mean outcomes between unprivileged and privileged groups = -0.000662


### 应用基于Adversarial Debiasing的处理中算法

In [6]:
tf.reset_default_graph()
# config = tf.ConfigProto()
# config.gpu_options.allow_growth = True
sess = tf.Session()

In [7]:
# Learn parameters with debias set to True
# 在去偏见设置为True时学习参数
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiased_classifier',
                          debias=True,
                          sess=sess)

In [ ]:
debiased_model.fit(dataset_orig_train)

In [ ]:
# 在训练去偏见模型后添加以下代码
# 导入可视化库

import seaborn as sns
import numpy as np

# 解决matplotlib中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 分析专家贡献
print("\n=== 分析MoE模型的专家贡献度 ===")

# 获取训练集和测试集的专家权重分析
train_analysis = debiased_model.analyze_expert_contributions(
    dataset_orig_train, 
    feature_names=dataset_orig_train.feature_names
)
test_analysis = debiased_model.analyze_expert_contributions(
    dataset_orig_test,
    feature_names=dataset_orig_test.feature_names
)

# 1. 可视化平均专家权重
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 训练集专家权重
ax1.bar(range(debiased_model.num_experts), train_analysis['average_weights'])
ax1.set_xlabel('专家编号')
ax1.set_ylabel('平均权重')
ax1.set_title('训练集：各专家平均权重分布')
ax1.set_xticks(range(debiased_model.num_experts))

# 测试集专家权重
ax2.bar(range(debiased_model.num_experts), test_analysis['average_weights'])
ax2.set_xlabel('专家编号')
ax2.set_ylabel('平均权重')
ax2.set_title('测试集：各专家平均权重分布')
ax2.set_xticks(range(debiased_model.num_experts))

plt.tight_layout()
plt.show()

In [ ]:
# 2. 特征-专家相关性热力图
correlation_matrix = np.array([
    train_analysis['feature_correlations'][feature] 
    for feature in dataset_orig_train.feature_names
])

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
            xticklabels=[f'专家{i}' for i in range(debiased_model.num_experts)],
            yticklabels=dataset_orig_train.feature_names,
            cmap='coolwarm',
            center=0,
            annot=True,
            fmt='.2f')
plt.title('特征与专家激活度的相关性')
plt.tight_layout()
plt.show()

In [ ]:
# # 3. 分析不同群体的专家激活模式
# # 获取特权和非特权群体的索引
# privileged_mask = dataset_orig_test.protected_attributes[:, 0] == 1
# unprivileged_mask = ~privileged_mask

# # 获取不同群体的门控输出
# gate_outputs_test = test_analysis['gate_outputs']
# privileged_gates = gate_outputs_test[privileged_mask]
# unprivileged_gates = gate_outputs_test[unprivileged_mask]

# # 计算每个群体的专家平均权重
# privileged_avg = np.mean(privileged_gates, axis=0)
# unprivileged_avg = np.mean(unprivileged_gates, axis=0)

# # 可视化对比
# fig, ax = plt.subplots(figsize=(10, 6))
# x = np.arange(debiased_model.num_experts)
# width = 0.35

# ax.bar(x - width/2, privileged_avg, width, label='特权群体', alpha=0.7)
# ax.bar(x + width/2, unprivileged_avg, width, label='非特权群体', alpha=0.7)

# ax.set_xlabel('专家编号')
# ax.set_ylabel('平均激活权重')
# ax.set_title('不同群体的专家激活模式对比')
# ax.set_xticks(x)
# ax.legend()

# plt.tight_layout()
# plt.show()

# # 4. 专家多样性分析
# # 计算专家输出的标准差，衡量专家的多样性
# gate_std = np.std(gate_outputs_test, axis=0)
# print("\n专家激活的标准差（多样性指标）：")
# for i, std in enumerate(gate_std):
#     print(f"专家{i}: {std:.4f}")

# # 5. 创建专家贡献度报告
# print("\n=== MoE模型可解释性报告 ===")
# print(f"模型使用了 {debiased_model.num_experts} 个专家")
# print("\n1. 专家专业化程度：")
# for i in range(debiased_model.num_experts):
#     # 找出与该专家最相关的特征
#     feature_corrs = [(feat, corr[i]) for feat, corr in train_analysis['feature_correlations'].items()]
#     top_feature = max(feature_corrs, key=lambda x: abs(x[1]))
#     print(f"   专家{i} 主要关注特征: {top_feature[0]} (相关性: {top_feature[1]:.3f})")

# print("\n2. 公平性分析：")
# print(f"   特权群体主要使用的专家: {np.argsort(privileged_avg)[-3:][::-1].tolist()}")
# print(f"   非特权群体主要使用的专家: {np.argsort(unprivileged_avg)[-3:][::-1].tolist()}")

# # 6. 保存详细分析结果
# results_df = pd.DataFrame({
#     '专家编号': list(range(debiased_model.num_experts)),
#     '平均权重_训练集': train_analysis['average_weights'],
#     '平均权重_测试集': test_analysis['average_weights'],
#     '特权群体权重': privileged_avg,
#     '非特权群体权重': unprivileged_avg,
#     '权重标准差': gate_std
# })

# print("\n专家权重详细统计：")
# print(results_df.round(4))

# # 可选：保存到CSV
# # results_df.to_csv('moe_expert_analysis.csv', index=False)

In [ ]:
# 将去偏见模型应用于测试数据
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)

In [ ]:
# 模型数据集的度量指标（不去偏见）

# 去偏见的模型数据集的度量指标
display(Markdown("#### Model - with debiasing - dataset metrics"))
metric_dataset_debiasing_train = BinaryLabelDatasetMetric(dataset_debiasing_train, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Train set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_train.mean_difference())

metric_dataset_debiasing_test = BinaryLabelDatasetMetric(dataset_debiasing_test, 
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)

print("Test set: Difference in mean outcomes between unprivileged and privileged groups = %f" % metric_dataset_debiasing_test.mean_difference())



# display(Markdown("#### Plain model - without debiasing - classification metrics"))
# print("Test set: Classification accuracy = %f" % classified_metric_nodebiasing_test.accuracy())
# TPR = classified_metric_nodebiasing_test.true_positive_rate()
# TNR = classified_metric_nodebiasing_test.true_negative_rate()
# bal_acc_nodebiasing_test = 0.5*(TPR+TNR)

# print("Test set: Balanced classification accuracy = %f" % bal_acc_nodebiasing_test)
# print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())
# print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
# print("Test set: Average odds difference = %f" % classified_metric_nodebiasing_test.average_odds_difference())
# print("Test set: Theil_index = %f" % classified_metric_nodebiasing_test.theil_index())
# print("Test set: Statistical parity difference = %f" % classified_metric_nodebiasing_test.statistical_parity_difference())
# print("Test set: Equalized Odds difference = %f" % classified_metric_nodebiasing_test.equalized_odds_difference())
# print("Test set: Equal opportunity difference = %f" % classified_metric_nodebiasing_test.equal_opportunity_difference())
# print("Test set: Disparate impact = %f" % classified_metric_nodebiasing_test.disparate_impact())


display(Markdown("#### Model - with debiasing - classification metrics"))
classified_metric_debiasing_test = ClassificationMetric(dataset_orig_test, 
                                                 dataset_debiasing_test,
                                                 unprivileged_groups=unprivileged_groups,
                                                 privileged_groups=privileged_groups)
print("Test set: Classification accuracy = %f" % classified_metric_debiasing_test.accuracy())
TPR = classified_metric_debiasing_test.true_positive_rate()
TNR = classified_metric_debiasing_test.true_negative_rate()
bal_acc_debiasing_test = 0.5*(TPR+TNR)

print("Test set: Statistical parity difference = %f" % classified_metric_debiasing_test.statistical_parity_difference())
print("Test set: Equalized Odds difference = %f" % classified_metric_debiasing_test.average_odds_difference())
print("Test set: Equal opportunity difference = %f" % classified_metric_debiasing_test.equal_opportunity_difference())
print("Test set: Disparate impact = %f" % classified_metric_debiasing_test.disparate_impact())